# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: ", metadata.name)
print("Description: ", metadata.description)
print("Published: ", getattr(metadata, "datePublished", "(not specified)"))
print("Version: ", getattr(metadata, "version", "(not specified)"))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate the `RecordSet` objects defined in the dataset, and for each, enumerate its fields and their `@id` values.

In [ ]:
# List available record sets and their structure
record_sets = [rs for rs in dataset.record_sets()]
if not record_sets:
    print("No record sets found in the dataset. Please check the schema definition.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs.id_}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {getattr(rs, 'description', '(none)')}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - Field Name: {field.name}")
            print(f"      @id: {field.id_}")
            print(f"      Data Type: {getattr(field, 'data_type', 'unknown')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Replace the placeholders below with the `@id` of the record set and relevant fields found in the previous overview.

In [ ]:
# Collect record set ids for extraction
record_set_ids = [rs.id_ for rs in dataset.record_sets()]
dataframes = {}

# Load each record set as a Pandas DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))
        print("-"*50)

# For main analysis, select the main record set (choose first or by inspecting previous cell)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    main_df = dataframes[main_record_set_id]
    print(f"Column names in '{main_record_set_id}':\n", main_df.columns.tolist())
    main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

Below, we select a numeric field (e.g., a PHQ-9 or GAD-7 score field) for filtering and normalization.

Replace `<numeric_field_id>` and `<group_field>` with the `@id`s of numeric and grouping fields found in your record set. You can use column lists printed above.

In [ ]:
# For illustration, choose field IDs for PHQ-9 and gender from your dataset
# Example: numeric_field = '@phq9_score', group_field = '@gender'

# Automatically try to find a numeric field (containing e.g. 'phq' or 'gad' or 'score') and a group field ('gender', 'village', etc.)
import re

def guess_field_id(df, possible_names):
    for col in df.columns:
        for key in possible_names:
            if re.search(key, col, re.IGNORECASE):
                return col
    return df.columns[0]  # fallback

if record_set_ids:
    df = dataframes[main_record_set_id]
    numeric_field = guess_field_id(df, ['phq', 'gad', 'psq', 'score', 'total'])
    group_field = guess_field_id(df, ['gender', 'sex', 'village', 'location'])

    print(f"Using numeric field: {numeric_field}")
    print(f"Using group field: {group_field}")

    # Filter out missing or non-numeric
    df = df.copy()
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print("Sample (original and normalized):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping and mean
    if group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we show the distribution of the chosen numeric field and a grouped bar plot if categorical data allows.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(main_df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Barplot of group means (if feasible)
if group_field in main_df.columns and main_df[group_field].nunique() < 15:
    plt.figure(figsize=(8,4))
    mean_by_group = main_df.groupby(group_field)[numeric_field].mean().sort_values()
    sns.barplot(x=mean_by_group.index, y=mean_by_group.values)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze the [FAIR² Mental Health Survey](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2) dataset using the Croissant metadata standard and the `mlcroissant` library. We:

- Explored dataset record sets and their structure with `@id` references
- Loaded main tabular data into pandas DataFrames
- Filtered and normalized a numeric mental health assessment score by `@id`
- Visualized data distributions and group-wise means

This notebook can be adapted for further, deeper analyses or reused with other Croissant-conformant datasets for rapid, FAIR AI-ready workflows.